[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-matthew-lam/relightable-3dgs/blob/main/run_project_in_colab.ipynb)

# **Relightable 3DGS Notebook**

## 1. Check the GPU / CUDA version

Run the following cell.

In [5]:
!nvidia-smi

## 2. Mount Google Drive

Run the following cell.

In [6]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone repo and install dependencies

Run the following cells.

In [25]:
import os

os.chdir("/content")  # always start from a fixed base, so this cell is safe to rerun in a live session
if not os.path.exists("/content/relightable-3dgs"):
    !git clone https://github.com/ml-matthew-lam/relightable-3dgs.git
%cd /content/relightable-3dgs
!git pull

In [8]:
!pip install -r requirements.txt

Now, restart the runtime, in case torch was somehow imported before installing `requirements.txt`. Then, re-run all the cells in the notebook until this point.

## 4. Copy dataset from Drive to Colab's local disk

Specify the Google Drive filepath to the dataset for `DRIVE_DATASET_ZIP` in the cell below. Then, run the cell.

In [9]:
import os
import shutil

# -------------- SPECIFY DATASET FILEPATH HERE -------------------
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkered_suzanne.zip"
# ----------------------------------------------------------------

LOCAL_DATASET_PATH = "/content/checkered_suzanne"

if not os.path.exists(LOCAL_DATASET_PATH):
    local_zip = "/content/checkered_suzanne.zip"
    shutil.copy(DRIVE_DATASET_ZIP, local_zip)
    shutil.unpack_archive(local_zip, "/content")
    os.remove(local_zip)
    print(f"copied and unzipped dataset to {LOCAL_DATASET_PATH}")
else:
    print(f"{LOCAL_DATASET_PATH} already exists, skipped copy")

## 5. Run the training script

In the cell below, specify the filepath you would like to use for the folder where checkpoints will be saved in Drive. Also change the values for arguments `--iters` (the number of training iterations) and `--num_init_points` (initial number of Gaussians) if desired. Then, run the cell.

In [ ]:
# --------- SPECIFY PATH TO CHECKPOINT FOLDER HERE ------------
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkpoints/lambertian_week3"
# ----------------------------------------------------------------

!python train.py \
  --data_dir /content/checkered_suzanne \
  --ckpt_dir "{DRIVE_CHECKPOINT_DIR}" \
  --iters 25000 \ 
  --num_init_points 100000

If runtime disconnects mid-run: reconnect, re-run cells until the end of step "4. Copy dataset from Drive to Colab's local disk," then specify `DRIVE_CHECKPOINT_DIR`, `--iters` and `num_init_points` in the cell below. Then run the cell below instead of the cell above.

In [ ]:
# --------- SPECIFY PATH TO CHECKPOINT FOLDER HERE ------------
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/checkpoints/lambertian_week3"
# ----------------------------------------------------------------

!python train.py \
  --data_dir /content/checkered_suzanne \
  --ckpt_dir "{DRIVE_CHECKPOINT_DIR}" \
  --iters 25000 \
  --num_init_points 100000 \
  --resume

## 6. Compare inferred renders to ground truth

In the following 4 cells, specify the paths to the Drive folders where you would like to save images of each of the side-by-side comparisons of inferred renders versus ground truths from the dataset. Also specify which checkpoint you want to use for the comparisons by editing the "step_xxxxxx.pt" portion of the `--ckpt` arguments. Then, run all 4 cells.

In [ ]:
# BEAUTY

# --------- SPECIFY WHERE TO SAVE THE BEAUTY COMPARISONS ------------
BEAUTY_COMPARISON_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_beauty"
# -------------------------------------------------------------------

!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_025000.pt" \
  --render_type beauty \
  --out_dir "{BEAUTY_COMPARISON_DIR}"

In [ ]:
# NORMALS

# --------- SPECIFY WHERE TO SAVE THE NORMAL COMPARISONS ------------
NORMAL_COMPARISON_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_normals"
# -------------------------------------------------------------------

!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_025000.pt" \
  --render_type normals \
  --out_dir "{NORMAL_COMPARISON_DIR}"

In [ ]:
# ALBEDO

# --------- SPECIFY WHERE TO SAVE THE ALBEDO COMPARISONS ------------
ALBEDO_COMPARISON_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_albedo"
# -------------------------------------------------------------------

!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_025000.pt" \
  --render_type albedo \
  --out_dir "{ALBEDO_COMPARISON_DIR}"

In [ ]:
# BEAUTY - test light (light5)

# ------- SPECIFY WHERE TO SAVE THE TEST LIGHT BEAUTY COMPARISONS --------
TEST_BEAUTY_COMPARISON_DIR = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/comparisons/week3_test_beauty"
# ------------------------------------------------------------------------

!python render_compare.py \
  --data_dir /content/checkered_suzanne \
  --ckpt "{DRIVE_CHECKPOINT_DIR}/step_025000.pt" \
  --render_type beauty \
  --test_light \
  --out_dir "{TEST_BEAUTY_COMPARISON_DIR}"

## 7. Render and view animation/video

The following cell saves a .webp animation/video demonstrating how the renders look from different views as well as different light positions. First, specify the location in Drive where you would like to save the animation/video. You can also adjust values for various arguments of `render_video.py` as desired. Then, run the cell.

In [ ]:
import os

# ------------ SPECIFY WHERE TO SAVE THE ANIMATION/VIDEO -------------
OUT_PATH = "/content/drive/MyDrive/Personal Projects/relightable-3dgs/videos/video_demo.webp"
# --------------------------------------------------------------------

os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

!python render_video.py \
  --ckpt_dir "{DRIVE_CHECKPOINT_DIR}" \
  --data_dir "{LOCAL_DATASET_PATH}" \
  --num_camera_positions 6 \
  --camera_radius 4.5 \
  --camera_elevation_deg 20 \
  --light_radius 4.5 \
  --light_cycles 2 \
  --transition_frames 60 \
  --dwell_frames 90 \
  --fps 24 \
  --out "{OUT_PATH}"

The cell above saves the animation/video to your specified filepath in Drive. If you would like to preview it right here in this notebook, run the following cell.

In [ ]:
import base64
from IPython.display import HTML

with open(OUT_PATH, "rb") as f:
    webp_b64 = base64.b64encode(f.read()).decode("utf-8")
HTML(f'<img src="data:image/webp;base64,{webp_b64}">')